<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week13_Transformer/bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT Pre-training and Fine-tuning

## Preparation

First, let's import necessary modules.

Note that utils.py includes some Blocks defined in the previous transformer notebook

In [10]:
!pip install d2l
!pip install mxnet
!pip install gluonnlp



In [11]:
import random, math
import numpy as np

import mxnet as mx
from mxnet import gluon, nd
from mxnet import npx              # ← new
npx.set_np()                       # ← new

from d2l import mxnet as d2l
import gluonnlp as nlp

from utils import PositionalEncoding, MultiHeadAttention 
from utils import AddNorm, PositionWiseFFN, EncoderBlock
from utils import train_loop, predict_sentiment

### BERT Model: it has only the encoder part of a transformer model
<img src="../img/transformer-bert.png" alt="architecture" width="450"/>

### Segment Embedding

Different from the transformer encoder, the BERT encoder has an additional embedding for segment information.

<img src="../img/bert-embed.png" alt="seg embedding" width="750"/>

Similar to the Transformer encoder defined in the previous section, the BERT encoder has embeddings for words and positions. The `EncoderBlock` contains position-wise feed-forward network and self-attention blocks to encode inputs. For BERT, the newly added segment embedding captures the segment information of the input sentence pairs, used for the next sentence prediction task.


In [12]:
import random
import math

import numpy as np
import mxnet as mx
from mxnet import gluon, nd
from mxnet.gluon import nn
import gluonnlp as nlp

class DotProductAttention(nn.Block): 
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    # query: (batch_size, #queries, d)
    # key: (batch_size, #kv_pairs, d)
    # value: (batch_size, #kv_pairs, dim_v)
    # mask: (batch_size, #queries, #kv_pairs)
    def forward(self, query, key, value, mask=None):
        d = query.shape[-1]
        # Use numpy-compatible batch matrix multiplication
        # Transpose key to get (batch_size, d, #kv_pairs)
        key_T = key.transpose((0, 2, 1))
        scores = mx.np.matmul(query, key_T) / math.sqrt(d)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores + mask * -1e9
        
        # Manual softmax implementation using NumPy operations
        # Subtract max for numerical stability
        scores_max = mx.np.max(scores, axis=-1, keepdims=True)
        exp_scores = mx.np.exp(scores - scores_max)
        attention_weights = exp_scores / mx.np.sum(exp_scores, axis=-1, keepdims=True)
        attention_weights = self.dropout(attention_weights)
        return mx.np.matmul(attention_weights, value)


class PositionalEncoding(gluon.nn.Block):
    def __init__(self, units, dropout=0, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self._max_len = max_len
        self._units = units
        self.embed = nn.Embedding(max_len, units)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        pos_seq = mx.np.expand_dims(mx.np.arange(X.shape[1]), axis=0)
        emb = self.embed(pos_seq)
        return self.dropout(X + emb)

class MultiHeadAttention(nn.Block):
    def __init__(self, units, num_heads, dropout, **kwargs):  # units = d_o
        super(MultiHeadAttention, self).__init__(**kwargs)
        assert units % num_heads == 0
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Dense(units, use_bias=False, flatten=False)
        self.W_k = nn.Dense(units, use_bias=False, flatten=False)
        self.W_v = nn.Dense(units, use_bias=False, flatten=False)

    # query, key, and value shape: (batch_size, num_items, dim)
    # mask shape is (batch_size, query_length, memory_length)
    def forward(self, query, key, value, mask):
        # Project and transpose from (batch_size, num_items, units) to
        # (batch_size * num_heads, num_items, p), where units = p * num_heads.
        query, key, value = [transpose_qkv(X, self.num_heads) for X in (
            self.W_q(query), self.W_k(key), self.W_v(value))]
        if mask is not None:
            # Replicate mask for each of the num_heads heads
            mask = nd.broadcast_axis(nd.expand_dims(mask, axis=1),
                                    axis=1, size=self.num_heads)
            # Get explicit dimensions for reshape
            batch_size, num_heads, query_len, key_len = mask.shape
            mask = mask.reshape((-1, query_len, key_len))
        output = self.attention(query, key, value, mask)
        # Transpose from (batch_size * num_heads, num_items, p) back to
        # (batch_size, num_items, units)
        return transpose_output(output, self.num_heads)

def transpose_qkv(X, num_heads):
    # Shape after reshape: (batch_size, num_items, num_heads, p)
    # Get explicit dimensions instead of using 0 (copy dimension)
    batch_size, num_items = X.shape[0], X.shape[1]
    X = X.reshape((batch_size, num_items, num_heads, -1))
    # Swap the num_items and the num_heads dimensions
    X = X.transpose((0, 2, 1, 3))
    # Merge the first two dimensions
    return X.reshape((-1, num_items, X.shape[-1]))

def transpose_output(X, num_heads):
    # A reversed version of transpose_qkv
    # Get explicit dimensions
    batch_num_heads, num_items, p = X.shape
    batch_size = batch_num_heads // num_heads
    X = X.reshape((batch_size, num_heads, num_items, p))
    X = X.transpose((0, 2, 1, 3))
    return X.reshape((batch_size, num_items, -1))

class PositionWiseFFN(nn.Block):
    def __init__(self, units, hidden_size, **kwargs):
        super(PositionWiseFFN, self).__init__(**kwargs)
        self.ffn_1 = nn.Dense(hidden_size, flatten=False)
        self.activation = nn.GELU()
        self.ffn_2 = nn.Dense(units, flatten=False)

    def forward(self, X):
        return self.ffn_2(self.activation(self.ffn_1(X)))

class AddNorm(nn.Block):
    def __init__(self, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm()

    def forward(self, X, Y):
        return self.norm(self.dropout(Y) + X)

def position_encoding_init(max_length, dim):
    X = nd.arange(0, max_length).reshape((-1,1)) / nd.power(
            10000, nd.arange(0, dim, 2)/dim)
    position_weight = nd.zeros((max_length, dim))

    position_weight[:, 0::2] = nd.sin(X)
    position_weight[:, 1::2] = nd.cos(X)
    return position_weight


class EncoderBlock(gluon.nn.Block):
    def __init__(self, units, hidden_size, num_heads, dropout, **kwargs):
        super(EncoderBlock, self).__init__(**kwargs)
        self.attention = MultiHeadAttention(units, num_heads, dropout)
        self.add_1 = AddNorm(dropout)
        self.ffn = PositionWiseFFN(units, hidden_size)
        self.add_2 = AddNorm(dropout)

    def forward(self, X, mask):
        Y = self.add_1(X, self.attention(X, X, X, mask))
        return self.add_2(Y, self.ffn(Y))

def train_loop(net, train_data, test_data, num_epoch, lr, ctx, loss_fn):
    trainer = gluon.Trainer(net.collect_params(), 'bertadam',
                            {'learning_rate': lr, 'wd':0.01},
                            update_on_kvstore=False)
    params = [p for p in net.collect_params().values() if p.grad_req != 'null']
    grad_clip = 1
    
    num_warmup_steps = 50
    step_num = 0
    num_train_steps = len(train_data) * num_epoch

    for epoch in range(num_epoch):
        accuracy = mx.metric.Accuracy()
        running_loss = 0
        for i, (inputs, seq_lens, token_types, labels) in enumerate(train_data):
            step_num += 1
    
            # learning rate schedule
            if step_num < num_warmup_steps:
                new_lr = lr * step_num / num_warmup_steps
            else:
                non_warmup_steps = step_num - num_warmup_steps
                offset = non_warmup_steps / (num_train_steps - num_warmup_steps)
                new_lr = lr - offset * lr
            trainer.set_learning_rate(new_lr)
            
            # CRITICAL FIX: Convert legacy arrays from batchify to NumPy arrays for BERT
            # The batchify operations produce legacy arrays, but BERT needs NumPy arrays
            inputs = inputs.as_np_ndarray()
            seq_lens = seq_lens.as_np_ndarray()  
            token_types = token_types.as_np_ndarray()
            labels = labels.as_np_ndarray()
            
            inputs = gluon.utils.split_and_load(inputs, ctx)
            seq_lens = gluon.utils.split_and_load(seq_lens, ctx)
            token_types = gluon.utils.split_and_load(token_types, ctx)
            labels = gluon.utils.split_and_load(labels, ctx)

            losses = []
            preds = [] 
            with mx.autograd.record():
                for inp, seq_len, token_type, label in zip(inputs, seq_lens, token_types, labels):
                    out = net(inp, token_type, seq_len)
                    loss = loss_fn(out, label.astype('float32'))
                    losses.append(loss)
                    preds.append(out)
            mx.autograd.backward(losses)
            for l in losses:
                running_loss += l.mean().item() / len(losses)
            # Gradient clipping
            trainer.allreduce_grads()
            nlp.utils.clip_grad_global_norm(params, 1)
            trainer.update(1)
    
            accuracy.update(labels, preds)
            if i % 25 == 0:
                print("Batch", i, "Acc", accuracy.get()[1],"Train Loss", running_loss/(i+1))
        print("Epoch {}, Acc {}, Train Loss {}".format(epoch, accuracy.get(), running_loss/(i+1)))
        evaluate(test_data, ctx, net)

def evaluate(test_data, ctx, net):
    accuracy = 0
    for i, (inputs, seq_lens, token_types, labels) in enumerate(test_data):
        # CRITICAL FIX: Convert legacy arrays from batchify to NumPy arrays for BERT
        inputs = inputs.as_np_ndarray()
        seq_lens = seq_lens.as_np_ndarray()
        token_types = token_types.as_np_ndarray()
        labels = labels.as_np_ndarray()
        
        inputs = gluon.utils.split_and_load(inputs, ctx)
        seq_lens = gluon.utils.split_and_load(seq_lens, ctx)
        token_types = gluon.utils.split_and_load(token_types, ctx)
        labels = gluon.utils.split_and_load(labels, ctx)
        for inp, seq_len, token_type, label in zip(inputs, seq_lens, token_types, labels):
            out = net(inp, token_type, seq_len)
            accuracy += (mx.np.argmax(out, axis=1).squeeze() == label).mean().copyto(mx.cpu()) / len(ctx)
        accuracy.wait_to_read()
    print("Test Acc {}".format(accuracy.item()/(i+1)))

def predict_sentiment(net, ctx, vocabulary, bert_tokenizer, sentence):
    ctx = ctx[0] if isinstance(ctx, list) else ctx
    max_len = 128
    padding_id = vocabulary[vocabulary.padding_token]

    transform = nlp.data.BERTSentenceTransform(bert_tokenizer, max_len, pad=False, pair=False)
    dataset = gluon.data.SimpleDataset([[sentence]])
    dataset = dataset.transform(transform)
    batchify_fn = nlp.data.batchify.Tuple(
        nlp.data.batchify.Pad(axis=0, pad_val=padding_id),
        nlp.data.batchify.Stack(),
        nlp.data.batchify.Pad(axis=0, pad_val=padding_id))
    predict_data = gluon.data.DataLoader(dataset, batchify_fn=batchify_fn,
                                         batch_size=1)

    for i, (inputs, seq_len, token_types) in enumerate(predict_data):
        # CRITICAL FIX: Convert legacy arrays from batchify to NumPy arrays for BERT
        inputs = inputs.as_np_ndarray().as_in_context(ctx)
        token_types = token_types.as_np_ndarray().as_in_context(ctx)
        seq_len = seq_len.as_np_ndarray().astype('float32').as_in_context(ctx)
        out = net(inputs, token_types, seq_len)
        label = mx.np.argmax(out, axis=1)
        return 'positive' if label.item() == 1 else 'negative'


### BERT Encoder 

In [13]:
class BERTEncoder(gluon.nn.Block):
    def __init__(self, vocab_size, units, hidden_size,
                 num_heads, num_layers, dropout, **kwargs):
        super(BERTEncoder, self).__init__(**kwargs)
        # segment_embed for segment information
        self.segment_embed = gluon.nn.Embedding(2, units)
        self.word_embed = gluon.nn.Embedding(vocab_size, units)
        self.pos_encoding = PositionalEncoding(units, dropout)
        self.blks = gluon.nn.Sequential()
        for i in range(num_layers):
            self.blks.add(EncoderBlock(units, hidden_size, num_heads, dropout))

    def forward(self, words, segments, mask, *args):
        X = self.word_embed(words) + self.segment_embed(segments)
        X = self.pos_encoding(X)
        for blk in self.blks:
            X = blk(X, mask)
        return X

### Using the BERT Encoder

Now let's test the BERTEncoder with a data batch of 2 sentence pairs, each with 8 words. Random integers are used to represent words for demonstration purpose. For segment information, we use 0 to indicate the word comes from the first sentence, 1 to indicate the second setence.

In [14]:
encoder = BERTEncoder(vocab_size=30000, units=768, hidden_size=3072,
                      num_heads=12, num_layers=12, dropout=0.1)
encoder.initialize()

num_samples, num_words = 2, 8
# random words for testing
words = mx.np.random.randint(0, 30000, (num_samples, num_words))
# the corresponding segment information for each word
segments = mx.np.array([[0,0,0,0,1,1,1,1],[0,0,0,1,1,1,1,1]])


encodings = encoder(words, segments, None)
print(encodings.shape) # (batch_size, num_words, units)

(2, 8, 768)


## Pre-training and Fine-tuning a BERT Model
<div style="width:900px;margin:auto;">

<img src="../img/bert-training-fine-tuning.png" alt="seg embedding" width="900"/>

</div>

## Definition of the tasks for fine-tuning 
<div style="width:900px;margin:auto;">

<img src="../img/bert-fine-tuning-tasks.png" alt="seg embedding" width="900"/>

</div>


### Pre-training Task 1: Next Sentence Classifier

Let us take a look at the first pre-training task: next sentence prediction. For this task, the encoding of the first token (the "[CLS]" token) is passed to a feed-forward network to make prediction.

Since next sentence prediction is a binary classification problem, we can use `SigmoidBinaryCrossEntropyLoss` as the loss function. In the following code block, we pass the encoding results to the `NSClassifier` to get the next sentence prediction. We use 1 as the label for true next sentence, and 0 otherwise. The prediction result and the label are then passed to the loss function for loss evaluation.

In [15]:
class NSClassifier(gluon.nn.Block):
    def __init__(self, units=768, **kwargs):
        super(NSClassifier, self).__init__(**kwargs)
        self.classifier = gluon.nn.Sequential()
        self.classifier.add(gluon.nn.Dense(units=units, flatten=False, activation='tanh'))
        self.classifier.add(gluon.nn.Dense(units=1, flatten=False))

    def forward(self, X, *args):
        X = X[:, 0, :]  # get the encoding of the first token
        return self.classifier(X)

ns_classifier = NSClassifier()
ns_classifier.initialize()

ns_pred = ns_classifier(encodings) # (batch_size, 1)
ns_label = mx.np.array([0, 1]) # 1 for true next setence, 0 otherwise
ns_loss_fn = gluon.loss.SigmoidBinaryCrossEntropyLoss()
ns_loss = ns_loss_fn(ns_pred, ns_label).mean()
print(ns_pred.shape, ns_loss.shape)

(2, 1) ()


### Pre-training Task 2: Masked Language Model (MLM) Decoder
<div style="width:900px;margin:auto;">

<img src="../img/bert-training-masked.png" alt="seg embedding" width="900"/>

</div>

Masked language modeling is one of the two pre-training tasks, where random positions are masked and the model needs to reconstruct the masked words. In the masked language model decoder, we first use `gather_nd` to pick the dense vectors representing words at masked position. Then a feed-forward network is applied on them, followed by a fully-connected layer to predict the unnormalized score for all words in the vocabulary.

In [16]:
class MLMDecoder(gluon.nn.Block):
    def __init__(self, vocab_size, units, **kwargs):
        super(MLMDecoder, self).__init__(**kwargs)
        self.decoder = gluon.nn.Sequential()
        self.decoder.add(gluon.nn.Dense(units, flatten=False))
        # Gaussian Error Linear Units as the activation function [4]
        self.decoder.add(gluon.nn.GELU())
        self.decoder.add(gluon.nn.LayerNorm())
        # classification layer for `vocab_size` classes
        self.decoder.add(gluon.nn.Dense(vocab_size, flatten=False))

    def forward(self, X, masked_positions, *args):
         # gather encodings at mask positions - NumPy compatible
        # masked_positions shape: (num_masked, 2) where each row is [batch_idx, position_idx]
        gathered = []
        for batch_idx, pos_idx in masked_positions:
            gathered.append(X[int(batch_idx), int(pos_idx)])
        X = mx.np.stack(gathered, axis=0)
        pred = self.decoder(X)
        return pred

### Using the Masked Language Model Decoder

In the following code block, we pass the encoding results to the `MLMDecoder` to get the masked language model prediction. We generate some random word indices as the label for demonstration purpose. For multi-class classification, we can use `SoftmaxCrossEntropyLoss` as the loss function. The prediction result and the label are then passed to the loss function for loss evaluation.

In [17]:
decoder = MLMDecoder(vocab_size=30000, units=768)
decoder.initialize()

# Fix: Use valid positions within the input dimensions (batch_size=2, seq_len=8)
# Fix: Use valid positions within the input dimensions (batch_size=2, seq_len=8)
mlm_positions = mx.np.array([[0,1],[1,4]])  # batch 0 pos 1, batch 1 pos 4
mlm_label = mx.np.array([100, 200])
mlm_pred = decoder(encodings, mlm_positions) # (batch_size, vocab_size)
mlm_loss_fn = gluon.loss.SoftmaxCrossEntropyLoss()
mlm_loss = mlm_loss_fn(mlm_pred, mlm_label).mean()
print(mlm_pred.shape, mlm_loss.shape)

(2, 30000) ()


## Here we skip the pre-training a BERT model from scratch

Below, we assume a pre-training BERT model is available from open sources.

## BERT Fine-tuning (Sentiment Analysis)

In this section, we fine-tune the BERT Base model for sentiment analysis on the IMDB dataset.

### BERT for Sentence Classification

Let's first take
a look at the BERT model
architecture for single sentence classification below:

<img src="../img/bert-sa.png" alt="bert sentiment analysis" width="750"/>


## References

[1] Devlin, Jacob, et al. "Bert:
Pre-training of deep
bidirectional transformers for language understanding."
arXiv preprint
arXiv:1810.04805 (2018).

[2] Dolan, William B., and Chris
Brockett.
"Automatically constructing a corpus of sentential paraphrases."
Proceedings of
the Third International Workshop on Paraphrasing (IWP2005). 2005.

[3] Peters,
Matthew E., et al. "Deep contextualized word representations." arXiv
preprint
arXiv:1802.05365 (2018).

[4] Hendrycks, Dan, and Kevin Gimpel. "Gaussian error linear units (gelus)." arXiv preprint arXiv:1606.08415 (2016).

For fine-tuning, we only need to initialize the last classifier layer from scratch. The other layers are already initialized from the pre-trained model weights.